# 09주차 · 임베딩과 딥러닝 기반 NLP
## Word2Vec · FastText · 문장 벡터 · RNN
**충북대학교 | 자연언어처리 | 임화정**

### 학습목표
- 희소 표현과 분산 표현의 차이, 분포 가설을 설명한다.
- CBOW/Skip-gram의 입력·예측 방향을 구분한다.
- gensim으로 Word2Vec/FastText를 학습하고 코사인 유사도를 계산한다.
- 학습 어휘와 OOV 벡터 생성의 차이를 설명한다.
- 평균 문장 벡터의 어순 손실과 RNN의 순차 처리를 비교한다.

### 실행 안내
Colab/Jupyter에서 위에서부터 실행합니다. CPU로 충분하며 GPU·JVM·사전학습 대형 모델은 필요 없습니다. Python 3.10 이상, numpy, pandas, gensim을 사용합니다. Colab에서 패키지가 없다면 아래 명령을 별도 코드 셀로 실행하세요. 설치에는 인터넷이 필요하며, 설치 후에도 import가 실패하면 런타임을 재시작하고 처음부터 실행하세요.
```python
%pip install -U gensim numpy scipy pandas
```

**데이터와 결과:** 말뭉치는 자체 작성한 교육용 템플릿 80문장입니다. 실제 뉴스/리뷰 데이터가 아니며, 작은 말뭉치의 유사어 순위를 언어 의미의 정답으로 볼 수 없습니다. 반복 학습은 고유 데이터의 다양성을 늘려주지 않습니다.

**학생용: TODO A/B/C/E·OOV 과제·응답란 포함**

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 60.1 MB/s  0:00:00


## 0. 환경·재현성
`workers=1`과 시드를 고정합니다. Python 해시, 라이브러리 버전, 연산 구현에 따라 런타임 간 결과가 달라질 수 있습니다. 엄격한 재현 실험은 프로세스 시작 전 PYTHONHASHSEED도 고정해야 합니다. Colab에서는 같은 실행 환경의 조건을 기록하는 것으로 시작하세요.

In [2]:
import sys
import numpy as np
import pandas as pd
import gensim
from gensim.models import Word2Vec, FastText
from IPython.display import display
SEED = 42
print('Python:', sys.version.split()[0])
print('numpy:', np.__version__, '| pandas:', pd.__version__, '| gensim:', gensim.__version__)


Python: 3.13.18
numpy: 2.4.6 | pandas: 2.2.3 | gensim: 4.4.0


## 1. 분포 가설과 교육용 말뭉치
분포 가설은 비슷한 문맥에 등장하는 단어가 비슷한 표현을 얻는다는 가설입니다. 같은 문맥에 반의어도 등장할 수 있으므로 **유사도=동의어 여부**가 아닙니다.

아래 문장은 이미 공백 단위로 구분된 토큰 예시입니다. 일반 한국어 원문에 공백 분할을 적용한 것과 다르며 형태소 분석 결과라고 주장하지 않습니다. 분야마다 4단어 × 5문맥 = 20문장, 전체 80문장입니다. 분야끼리 문맥이 인위적으로 분리되어 있다는 편향을 기록하세요.

In [6]:
groups = {
    '동물': (
        ['강아지', '개', '고양이', '토끼'],
        [
          '나는 {w} 를 좋아한다', '{w} 는 귀여운 동물 이다',
          '집 에서 {w} 와 함께 지낸다', '{w} 에게 먹이 를 준다',
          '공원 에서 {w} 와 산책 한다'
        ]
    ),
    '음식': (
        ['사과', '바나나', '딸기', '포도'],
        [
          '나는 {w} 를 먹는다', '{w} 는 맛있는 과일 이다',
          '시장 에서 신선한 {w} 를 산다', '{w} 로 주스 를 만든다',
          '아침 에 달콤한 {w} 를 먹는다'
        ]
    ),
    '영화': (
        ['영화', '드라마', '공연', '연극'],
        [
          '나는 {w} 를 감상 한다', '{w} 의 배우 연기 가 훌륭하다',
          '{w} 의 이야기 가 재미있다', '친구 와 {w} 를 보고 감동 했다',
          '{w} 의 결말 과 연출 이 좋다'
        ]
    ),
    '제품': (
        ['휴대폰', '노트북', '태블릿', '컴퓨터'],
        [
          '나는 {w} 를 사용 한다', '{w} 의 화면 이 밝다',
          '{w} 의 배터리 가 오래 간다', '새로운 {w} 의 성능 이 좋다',
          '{w} 의 가격 과 품질 을 비교 한다'
        ]
    )
}
records = []
for domain, (words, sentence_templates) in groups.items():
    for word in words:
        for sentence_template in sentence_templates:
            records.append({'domain': domain, 'text': sentence_template.format(w=word)}) # placeholder에 값 대입해 text 완성하고, records 배열에 추가
corpus_df = pd.DataFrame(records)
sentences = [text.split() for text in corpus_df['text']]
assert len(sentences) == 80
assert not corpus_df['text'].duplicated().any() # 중복된 text가 하나도 없는지 확인(duplicated는 중복 True/False series를 반환하고, any는 해당 series에 True가 없는지 확인)
vocabulary = sorted({word for sent in sentences for word in sent})
print('문장 수:', len(sentences), '| 고유 토큰 수:', len(vocabulary)) # 문장이 template이 정해져 있기 때문에, 문장 수보다 고유 토큰 수가 더 적을 수도 있음.
display(corpus_df.groupby('domain').size().to_frame('sentences')) # 도메인 기준으로 묶어서, 각 도메인의 행 개수를 표시하고, 해당 series를 dataframe으로 만듦.
print('첫 문장 토큰:', sentences[0])


문장 수: 80 | 고유 토큰 수: 75


,sentences
domain,
동물,20
영화,20
음식,20
제품,20


첫 문장 토큰: ['나는', '강아지', '를', '좋아한다']


## 2. 중심 단어와 문맥 쌍
Skip-gram은 중심 → 문맥, CBOW는 주변 문맥 → 중심을 예측합니다. `window=2`이면 경계에서는 주변 단어 수가 줄어듭니다. 아래는 **고정 윈도우 개념 예시**이며 gensim 기본 학습은 윈도우를 확률적으로 줄일 수 있습니다.

### 실습 A
중심 위치를 제외하고 최대 window 거리의 단어를 수집하는 함수를 완성하세요.

In [9]:
def context_words(tokens, center_index, window=2):
    # TODO A: 좌우 범위를 계산하고 중심 위치를 제외하세요.
    naive_start = center_index - window # 나이브한 계산값
    naive_end = center_index + window + 1 # 1은 슬라이싱 시 끝처리용

    start = naive_start if naive_start >= 0 else 0 # 경계에서는 경계를 넘어가지 않도록 처리
    end = naive_end if naive_end <= len(tokens) else len(tokens)

    return tokens[start:center_index] + tokens[center_index+1:end]

example = ['나는', '귀여운', '강아지', '와', '산책', '한다']
contexts = context_words(example, 2, window=2)
print('CBOW:', contexts, '→', example[2])
print('Skip-gram:', [(example[2], c) for c in contexts])
print('완료 확인:', contexts == ['나는', '귀여운', '와', '산책'])


CBOW: ['나는', '귀여운', '와', '산책'] → 강아지
Skip-gram: [('강아지', '나는'), ('강아지', '귀여운'), ('강아지', '와'), ('강아지', '산책')]
완료 확인: True


## 3. Word2Vec 학습
입력 임베딩과 출력 임베딩이라는 두 가중치 행렬을 학습합니다. `wv`는 일반적으로 사용하는 입력 단어 벡터입니다. negative sampling의 negative는 '부정 감성'이 아니라 학습용 대조 단어 쌍을 뜻합니다.

|설정|의미|
|---|---|
|vector_size|단어 벡터 차원|
|window|최대 문맥 거리|
|min_count|최소 토큰 빈도|
|sg|0=CBOW, 1=Skip-gram|
|negative|정답 쌍당 대조 단어 수|
|epochs|학습 반복 횟수|
|sample=0|이 작은 예제에서 빈번한 토큰 subsampling을 끔|

### 실습 B
CBOW와 동일 조건으로 Skip-gram 모델을 만들되 sg를 변경하세요. 학생용 기본값은 CBOW이므로 수정 전에는 Skip-gram이라고 해석하지 마세요.

In [49]:
COMMON = dict(vector_size=32, window=2, min_count=1, workers=1,
              seed=SEED, epochs=150, negative=5, sample=0)
cbow = Word2Vec(sentences=sentences, sg=0, **COMMON)
# TODO B: sg를 수정해 Skip-gram으로 만드세요.
skipgram = Word2Vec(sentences=sentences, sg=1, **COMMON)
print('CBOW sg:', cbow.sg, '| Skip-gram sg:', skipgram.sg)
print('벡터 배열:', skipgram.wv.vectors.shape)
print('강아지 벡터 앞 8차원:', skipgram.wv['강아지'][:8])
print('완료 확인:', skipgram.sg == 1)


CBOW sg: 0 | Skip-gram sg: 1
벡터 배열: (75, 32)
강아지 벡터 앞 8차원: [ 0.07493291 -0.72868663  0.7271582   0.43329352 -0.6473702  -0.8821525
 -0.79962975 -0.1034485 ]
완료 확인: True


## 4. 코사인 유사도와 유사어 탐색
코사인 유사도는 내적/(두 벡터의 길이 곱)입니다. 1은 같은 방향, -1은 반대 방향, 0은 직교를 뜻합니다. 0이 모든 의미 관계의 부재를 증명하는 것은 아닙니다. 벡터 길이가 0이면 코사인은 정의되지 않습니다.

### 실습 C
직접 계산한 코사인을 gensim 결과와 비교하세요.

In [50]:
def cosine_similarity_manual(a, b):
    # TODO C: 분모가 0이면 np.nan, 그렇지 않으면 코사인 공식을 반환하세요.
    size_a = 0
    size_b = 0
    for i in range(len(a)): # a, b의 차원은 같으므로 한번에 진행 가능
      size_a = size_a + a[i]*a[i]
      size_b = size_b + b[i]*b[i]

    if size_a == 0 or size_b == 0:
      return np.nan

    return np.dot(a, b) / ((size_a)**0.5 * (size_b)**0.5)

manual = cosine_similarity_manual(skipgram.wv['강아지'], skipgram.wv['개'])
reference = float(skipgram.wv.similarity('강아지', '개'))
print('직접 계산:', manual, '| gensim:', reference)
print('완료 확인:', np.isclose(manual, reference, atol=1e-6))
rows = []
for name, model in [('CBOW', cbow), ('Skip-gram', skipgram)]:
    for word, score in model.wv.most_similar('강아지', topn=5):
        rows.append({'model': name, 'neighbor': word, 'cosine': score})
display(pd.DataFrame(rows))


직접 계산: 0.99690115 | gensim: 0.9969011545181274
완료 확인: True


,model,neighbor,cosine
0,CBOW,개,0.997906
1,CBOW,고양이,0.997396
2,CBOW,토끼,0.996463
3,CBOW,지낸다,0.718324
4,CBOW,시장,0.710639
5,Skip-gram,고양이,0.999099
6,Skip-gram,토끼,0.997583
7,Skip-gram,개,0.996901
8,Skip-gram,지낸다,0.763946
9,Skip-gram,준다,0.671188


### 해석 질문
- 유사어 상위 결과가 기대와 다르다면 무엇을 확인할까요?
  - 데이터의 규모와 다양성, 벡터공간의 크기 등을 체크해야 합니다.
- vector_size를 키우면 품질이 반드시 좋아질까요?
  - 품질은 어느 수준까지 좋아지다가, 반대로 떨어집니다.
  - 단순 코사인유사도만 보면 size가 32였을 때 강아지와 개의 코사인 유사도는 0.9969였고, 300으로 늘리니 0.9979가 되었습니다.
  - 그러나 유사도가 높은 단어 5개를 뽑았을 때는 size가 32였을 때 개, 고양이, 토끼, 지낸다, 시장(CBOW), 고양이, 토끼, 개, 지낸다, 준다(Skip gram) 등 실제로 유사도가 있는 단어들 위주로 나온 반면, size를 300으로 올리니 개, 고양이, 토끼, 딸기, 포도(CBOW), 고양이, 토끼, 개, 지낸다, 시장(Skip gram) 등 실제로 유사도가 있진 않은 단어들도 일부 포함되었습니다.
- '왕-남자+여자≈여왕'은 언제나 재현되나요?
  - 데이터가 부족한 경우, 과적합으로 인해 이와 같은 고차원적 유추 연산은 전혀 재현되지 않습니다.
  - 또한, 해당 예시는 초대형 데이터셋에서 발견된 몇 가지 대표적인 사례일 뿐 임의의 모든 단어 쌍에 대해 완벽한 유추가 항상 성립하지는 않습니다.


## 5. 문자 n-gram과 FastText
FastText는 단어와 문자 subword 정보를 활용합니다. 문자 n-gram은 형태소 분석이나 BERT의 WordPiece와 다른 단위입니다. 아래 함수는 Unicode 문자 단위 **개념 예시**이며 gensim 내부의 byte 처리·hash bucket을 재현하지 않습니다.

`<강아지>`를 n=2로 분리하면 `<강`, `강아`, `아지`, `지>`이며 n=3이면 `<강아`, `강아지`, `아지>`입니다. 경계 기호도 포함합니다.

In [32]:
def character_ngrams(word, min_n=2, max_n=3):
    bounded = '<' + word + '>'
    return {n: [bounded[i:i+n] for i in range(len(bounded)-n+1)]
            for n in range(min_n, max_n+1)}
print(character_ngrams('강아지'))
fasttext = FastText(sentences=sentences, sg=1, min_n=2, max_n=4,
                    bucket=5000, **COMMON)


{2: ['<강', '강아', '아지', '지>'], 3: ['<강아', '강아지', '아지>']}


### 실습 D · OOV 확인
`강아지랑`은 이 말뭉치에 없지만 subword 조합으로 FastText 벡터를 요청할 수 있습니다. `key_to_index`는 실제 학습 단어 목록입니다. FastText에서 `word in wv`는 벡터 조회 가능성을 포함할 수 있으므로 **학습 어휘 판정에는 key_to_index**를 사용하세요.

벡터 생성 가능≠의미 정확성 보장입니다. 공유 subword가 적거나 덜 학습되었으면 신뢰할 수 없습니다.

In [46]:
oov_word = '원숭이'
assert oov_word not in vocabulary
rows = []
for name, model in [('Word2Vec', skipgram), ('FastText', fasttext)]:
    learned = oov_word in model.wv.key_to_index
    try:
        vector = model.wv[oov_word]
        available, norm = True, float(np.linalg.norm(vector))
    except KeyError:
        available, norm = False, np.nan
    rows.append({'model': name, 'learned_word': learned,
                 'vector_available': available, 'vector_norm': norm})
display(pd.DataFrame(rows))
assert not rows[0]['vector_available'] and rows[1]['vector_available']
print('FastText OOV 유사어(정답 아님):', fasttext.wv.most_similar(oov_word, topn=3))


,model,learned_word,vector_available,vector_norm
0,Word2Vec,False,False,NaN
1,FastText,False,True,1.262471


FastText OOV 유사어(정답 아님): [('먹이', 0.8032889366149902), ('고양이', 0.7695324420928955), ('개', 0.7660882472991943)]


### 응답 D
`강아지랑`의 학습어휘 포함 여부와 벡터 조회 가능 여부를 각각 적으세요. 완전히 다른 미등록 단어 하나를 직접 골라 같은 방식으로 확인하고, 의미 품질을 보장할 수 없는 이유를 쓰세요.
- `강아지랑`은 학습어휘에 포함되지 않았습니다. Word2Vec에서는 벡터 조회가 실패했지만, FastText에서는 문자 n-gram 덕분에 벡터 조회에는 성공했습니다.
- `원숭이`의 경우 학습어휘에 포함되지 않았습니다. Word2Vec에서는 벡터 조회가 실패했지만, FastText에서는 벡터 조회에는 성공했습니다. 이때 유사어로 `먹이`가 등장한 것으로 볼 때, 의미 품질을 보장할 수는 없습니다.

## 6. 평균 문장 벡터
문장의 단어 벡터들을 평균합니다. Word2Vec에서는 OOV를 제외하고, FastText에서는 조회 가능한 OOV를 포함합니다. **모두 누락되거나 빈 문장이면 0벡터**를 반환하며, 길이0 벡터를 유사도 계산에 무조건 넣지 않습니다. coverage도 같이 기록합니다.

두 문장의 토큰 다중집합이 같으면 평균 벡터가 같습니다. 따라서 주어/목적어 순서로 의미가 달라져도 구분하지 못합니다.

### 실습 E
수집한 벡터를 축0 방향으로 평균하는 부분을 완성하세요.

In [47]:
def mean_sentence_vector(text, keyed_vectors):
    tokens = text.split()
    vectors = []
    for token in tokens:
        try:
            vectors.append(keyed_vectors[token])
        except KeyError:
            pass
    coverage = len(vectors) / len(tokens) if tokens else 0.0
    if not vectors:
        return np.zeros(keyed_vectors.vector_size, dtype=np.float32), coverage
    # TODO E: 아래 기본 0벡터를 vectors의 평균으로 바꾸세요.
    return np.mean(vectors, axis=0), coverage

sentence_a = '강아지 가 고양이 를 좋아한다'
sentence_b = '고양이 가 강아지 를 좋아한다'
vec_a, coverage_a = mean_sentence_vector(sentence_a, skipgram.wv)
vec_b, coverage_b = mean_sentence_vector(sentence_b, skipgram.wv)
print('coverage:', coverage_a, coverage_b)
print('어순 변경 후 평균 벡터 같음:', np.allclose(vec_a, vec_b))
print('완료 확인(영벡터 아님):', np.linalg.norm(vec_a) > 0)
empty_vec, empty_coverage = mean_sentence_vector('', skipgram.wv)
unknown_vec, unknown_coverage = mean_sentence_vector('미등록단어xyz', skipgram.wv)
print('빈/OOV 문장 coverage:', empty_coverage, unknown_coverage)


coverage: 1.0 1.0
어순 변경 후 평균 벡터 같음: True
완료 확인(영벡터 아님): True
빈/OOV 문장 coverage: 0.0 0.0


## 7. 정적 임베딩에서 순차 모델로
Word2Vec/FastText에서는 같은 단어 표기가 문맥과 무관하게 같은 벡터를 갖습니다. 예를 들어 '배'의 과일/선박 의미를 표기 하나로 자동 분리하지 않습니다. 평균 pooling은 순서까지 버립니다.

RNN은 `h_t = tanh(W_h @ h_{t-1} + W_x @ x_t + b)`로 이전 상태와 현재 입력을 결합합니다. 아래는 **시드 고정 랜덤 가중치로 순서 영향을 보여주는 계산 데모**입니다. 가중치를 학습하지 않았고 감성 분류기도 아닙니다. 상태가 다르다는 사실은 의미를 정확히 이해했다는 뜻이 아닙니다.

In [48]:
rng = np.random.default_rng(SEED) # 난수 생성기(시드 고정)
DIM = skipgram.wv.vector_size
HIDDEN = 8
W_x = rng.normal(0, 0.15, (HIDDEN, DIM)) # 정규 분포 따르는 난수를 생성. 평균은 0, 표편은 0.15, 행렬의 형태는 (HIDDEN, DIM)
W_h = rng.normal(0, 0.15, (HIDDEN, HIDDEN)) # W_x는 입력 가중치 행렬(현재 단어의 의미 반영), W_h는 이전 기억을 얼마나 보존할지 결정하는 순환 가중치 행렬
bias = np.zeros(HIDDEN)

def rnn_demo(text, keyed_vectors):
    h = np.zeros(HIDDEN) # 이전 기억 초기화
    states = []
    for token in text.split():
        try:
            x = keyed_vectors[token]
        except KeyError:
            x = np.zeros(DIM)
        h = np.tanh(W_h @ h + W_x @ x + bias)
        states.append(h.copy())
    return h, np.array(states)

h_a, states_a = rnn_demo(sentence_a, skipgram.wv)
h_b, states_b = rnn_demo(sentence_b, skipgram.wv)
print('최종 상태 A:', np.round(h_a, 4))
print('최종 상태 B:', np.round(h_b, 4))
print('순서 변경 후 상태 같음:', np.allclose(h_a, h_b))
print('상태 간 L2 거리:', float(np.linalg.norm(h_a-h_b)))
assert states_a.shape == (len(sentence_a.split()), HIDDEN)
assert not np.allclose(h_a, h_b)


최종 상태 A: [ 0.2843  0.7361  0.2301 -0.5499 -0.0782 -0.7582 -0.3796  0.1692]
최종 상태 B: [ 0.2841  0.7346  0.228  -0.5474 -0.0805 -0.7585 -0.3794  0.166 ]
순서 변경 후 상태 같음: False
상태 간 L2 거리: 0.005377619879129775


### RNN의 한계와 10주차 연결
- 순차 처리 때문에 훈련 병렬화가 제한됩니다.
- 긴 문장의 정보 보존과 학습에서 장기 의존성·기울기 소실 문제가 생길 수 있습니다.
- LSTM/GRU는 게이트로 이를 완화하지만 모든 문제를 해결하지는 않습니다.
- 다음 주 Attention/Transformer에서는 토큰 간 관계를 직접 계산하고 위치 정보를 추가합니다.
- Doc2Vec은 문서 벡터를 함께 학습하는 별도 방법이며, 단어 평균이나 완전한 어순 이해와 같지 않습니다.

## 8. 임베딩을 분류에 쓸 때의 데이터 누수
이번 노트북은 **표현 학습·탐색** 실습이며 라벨 분류 성능을 평가하지 않습니다. 임베딩을 분류기에 연결해 평가한다면:
1. 원문을 train/validation/test로 먼저 분할합니다.
2. 직접 학습한 임베딩은 train 텍스트만으로 학습합니다.
3. validation으로 차원/window/분류기 등을 결정합니다.
4. 확정 후 test를 한 번 평가합니다.
5. 외부 사전학습 임베딩은 출처·라이선스·학습 데이터 겹침 위험을 기록합니다.

test 문장을 라벨 없이 임베딩 학습에 사용해도 일반적인 독립 평가에서는 누수입니다. transductive 설정이라면 이를 별도로 명시해야 합니다.

## 9. 형성평가·보고서 과제
### 확인 질문
1. CBOW/Skip-gram의 입력과 예측 대상을 쓰세요.

- CBOW는 여러 문맥 단어 벡터들을 입력으로 받아 중심 단어를 예측합니다.
- Skip-gram은 중심 단어를 입력으로 받아 주변 문맥 단어들을 예측합니다.
2. cosine=0의 기하학적 뜻은 무엇인가요?
- 두 벡터가 직교한다는 의미입니다.
3. FastText OOV 조회가 의미 품질을 보장하지 않는 이유는?
- 완전히 무관한 외래어나 오탈자가 기존 단어의 subword와 우연히 겹칠 경우, 잘못된 방향의 임베딩 벡터로 심각하게 왜곡될 수 있습니다.
4. 평균 문장벡터가 어순을 구분하지 못하는 이유는?
- 단어의 순서를 보존하지 않고 단순히 산술 평균하기 때문입니다.
5. 랜덤 RNN 상태가 다르면 문장 의미를 이해했다고 볼 수 있나요?
- 학습을 하지 않았기 때문에, 수학적인 구조의 특성에 따라 상태만 다를 뿐, 의미를 이해했다고 볼 수는 없습니다.
6. 분류 실험에서 test를 임베딩 학습에 넣어도 되나요?
- 데이터 누수에 해당하므로, 넣으면 안됩니다.

### 과제
- 실습 A/B/C/E TODO를 완성하고, D의 OOV 해석을 작성하세요.
- 단어 두 개에 대해 CBOW/SG 유사어 순위를 비교하고 차이를 설명하세요.
- **선택 확장:** window=1과 window=3을 같은 조건으로 학습하여 순위 변화를 기록하세요. 최적 성능을 주장하려면 별도 평가가 필요합니다.
- 평균 pooling과 RNN의 결과를 비교하되 학습 여부와 의미 한계를 구분하세요.
- 말뭉치의 분야별 템플릿 편향, 작은 규모, 어절/형태소 차이 등 한계를 적으세요.
  -  말뭉치는 분야별로 템플릿이 편향되어 있습니다. 예를 들어, '시장'이라는 단어는 꼭 음식이라는 도메인과 엮이지 않아도 되지만, 현재는 음식이라는 도메인에 엮여 있습니다. 또한 데이터셋은 규모가 작아, 학습의 품질이 높지 않습니다. 현재는 임시적으로 명사와 조사 사이를 띄어쓰기로 분리하여 각각을 단어로 만들어주었는데, 이는 추후 형태소 분석을 통해 개선해야 합니다.

### 응답란
- 중심/문맥 방향: CBOW는 문맥 -> 중심, Skip gram은 중심 -> 문맥 방향입니다.
- 직접 코사인 계산: 직접 계산했을 때, Word2Vec에서 제공하는 wv.similarity와 차이가 없었습니다.
- 유사어 비교와 한계: '강아지'에 대해 유사어 순위를 비교했는데, CBOW는 포도, 딸기 등 연관성이 떨어지는 단어들이 등장한 반면, FastText는 지낸다 등 비교적 연관성이 있는 단어들이 등장했습니다.
- OOV 실험 및 해석: '강아지랑', '원숭이'에 대해 유사어 순위를 비교했는데 '강아지랑'의 경우 동물들이 유사어 순위 상위권에 오른 반면, '원숭이'의 경우 '먹이'라는 연관성이 떨어지는 단어가 유사어 순위 상위권에 올랐습니다. FastText를 이용하면 OOV에 대해서도 임베딩 벡터를 추출할 수 있으나, 그것이 의미의 품질이 높다고 볼 수는 없습니다.
- 어순·평균·RNN 비교: 평균 pooling은 어순을 학습하지 못합니다. RNN은 어순을 학습할 수 있으나, 해당 예시에서는 학습을 진행하지 않았기 때문에 의미를 이해했다고 볼 수는 없습니다.
- 데이터 누수 방지 계획: seed를 고정한 상태로 데이터를 train / test로 분리하고, 분류 실험에서 test를 임베딩 학습에 사용하지 않습니다.

### 제출 체크리스트
- [x] TODO 완료 후 처음부터 순서대로 실행
- [x] 실행 결과 저장 / 시드·버전·하이퍼파라미터 기록
- [x] 교육용 자체작성 말뭉치임을 명시
- [x] OOV 학습어휘와 조회 가능성 구분
- [x] 의미 품질·분류 성능을 과장하지 않음

제출 예시: `학번_이름_09주차.ipynb` 및 해석을 적은 README. GitHub/LMS 제출.

## 참고 자료와 원자료 보완
- 제공 「Part3 딥러닝 9–12주차 강의안」 9주차(5–9쪽)
- 제공 「자연언어처리 강의계획서」 9주차
- [gensim Word2Vec 문서](https://radimrehurek.com/gensim/models/word2vec.html)
- [gensim FastText 문서](https://radimrehurek.com/gensim/models/fasttext.html)

문자 n-gram 경계 예시와 RNN의 h_{t-1} 표기를 명확히 했습니다. 원자료의 소량 말뭉치만으로 의미 품질을 보장하지 않으며, 형태소 분석은 선택 확장으로 구분합니다. 학습 결과와 랜덤 RNN 개념 데모를 구별합니다.